# L3d: Comparing Sorting Algorithms and Their Scaling
In this lab, we finish [our own bubble-sort implementation](src/Compute.jl), listen to it work, and then compare its average runtime against a recursive [Quicksort](https://en.wikipedia.org/wiki/Quicksort) implementation and [Julia's built-in sort function](https://docs.julialang.org/en/v1/base/sort/#Base.sort) using [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl).

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Finish an implementation against its contract:__ Complete a partly written in-place routine so that it meets a stated contract and reproduces a reference result. Reason about what an algorithm costs from code you wrote yourself rather than from a description of it.
> * __Test an implementation before timing it:__ Establish that a routine reproduces a reference result before measuring how fast it runs, because timing incorrect code tells you nothing. Check a mutating routine and its non-mutating wrapper against their separate contracts, since one rewrites its argument and the other must leave it untouched.
> * __Measure scaling and decide between building and buying:__ Benchmark one operation across growing inputs and read the growth rate separately from the constant factor in front of it. Comparing our own implementations against the library routine helps you make a concrete build-versus-buy decision.

Let's get started!
___

## Algorithms
We time two sorting algorithms that take different approaches, and then compare both against the library.

* __Bubblesort__ passes over the list, compares neighbors, and swaps any pair that is out of order. Each pass carries the largest remaining value to the end, which is where the name comes from. [Read the details](CHEME-5800-L3d-Algorithm-Bubblesort-Fall-2026.ipynb).
* __Quicksort__ picks a pivot, splits the list around it, and sorts each side the same way. How the pivot is chosen is what decides whether it runs fast or slowly. [Read the details](CHEME-5800-L3d-Algorithm-Quicksort-Fall-2026.ipynb).

The lab has three tasks:

1. Finish our `bubblesort!(...)` and test it on a small array.
2. Test it again on a larger array, then benchmark it to set a baseline.
3. Measure our Quicksort and Julia's built-in sort against that baseline.
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The course environment also loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl); see [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/). This lab does not need it. The sorting implementations we benchmark live in [`src/Compute.jl`](src/Compute.jl), and the timing comes from [the BenchmarkTools.jl package](https://github.com/JuliaCI/BenchmarkTools.jl). The bubble-sort tones load through [the WAV.jl package](https://github.com/dancasimiro/WAV.jl), and the `sounds/` folder holds the 128 tone files described in [`sounds/README.md`](sounds/README.md).

### Constants
Let's set the constants this lab uses. The comment next to each value says what it is and what it controls.

In [ ]:
max_number_of_trials = 10; # dimensionless exponent; the largest vector holds 2^10 = 1024 elements
number_of_items_per_trial = [2^i for i ∈ 1:max_number_of_trials]; # vector lengths 2, 4, ..., 1024; an array comprehension, yet another iteration pattern
play_sounds = false; # true or false; flip to true in class to hear each bubble-sort pass

___

## Task 1: Complete the bubble-sort implementation
In this task, we finish the bubble sort ourselves, since the timing comparisons later are useful only if we understand the code being timed. The [`src/Compute.jl`](src/Compute.jl) file holds the `L3dSorting` module, and [our `bubblesort!(...)` function](src/Compute.jl) in it throws an error until you complete it. The non-mutating [`bubblesort(...)` wrapper](src/Compute.jl) copies its input and passes the copy to the mutating version, so finishing one finishes both.

> __What to write:__
>
> * __TODO 1:__ Loop over passes `1` through `length(values)`. At the top of each pass, call [the private `_play_sound(...)` helper](src/Sounds.jl) if a sound library was supplied.
> * __TODO 2:__ Inside each pass, sweep positions `1` through `length(values) - pass`, counted by [the `length(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.length), and swap any neighboring pair that [the `isless(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.isless) says is out of order. That is the ordering [Julia's `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Base.sort) uses.
> * __TODO 3:__ Track whether the pass swapped anything, stop early when a full pass changes nothing, and return the sorted vector.

### Reloading your edits
Work in the source file and save it, then re-run the setup cell in the Setup, Data, and Prerequisites section. The reload is silent, so do not wait for a confirmation message. The notebook calls the module's functions by their qualified names, such as `L3dSorting.bubblesort(...)`, which is why the reloaded module takes effect without restarting the kernel. Then re-run the test cell.

### Sound or silent
The `sounds/` folder holds 128 short tones, one per integer value, rising in pitch with the value, and [`sounds/README.md`](sounds/README.md) records where they came from. With the `play_sounds` constant set to `true`, the `sound_library::Union{Nothing, Dict{Int64, Tuple{Matrix{Float64}, Float32}}}` variable loads all of them, and sorting an integer array drawn from `1:128` then plays the array at the top of every pass. A shuffled array sounds jumbled, and a finished sort plays a rising scale.

The `CHEME5800_L3D_ROOT` constant below is this folder's path, set by [`Include.jl`](Include.jl), so the load works wherever the notebook was launched from. Let's load the library, or stay silent:

In [ ]:
sound_library = play_sounds ? L3dSorting.load_sound_library(joinpath(CHEME5800_L3D_ROOT, "sounds")) : nothing; # 128 tones, or silent

You are finished when this test passes: a sorted copy must match the reference ordering exactly on the shuffled input we give it. When the sound library is loaded, this check also plays the array at the top of every pass. Does it pass?

In [ ]:
let
    audible_demo = rand(1:128, 25); # integer values 1:128, so each maps to a tone
    @test L3dSorting.bubblesort(audible_demo; sounds = sound_library) == sort(audible_demo)
end

A passing test says the completed implementation agrees with the library on this input. The timing comparisons depend on that: what we measure is the cost of a correct sort, not of incorrect code.
___

## Task 2: Establish the bubble-sort performance baseline
In this task, we test the completed bubble sort on a larger random input and then time it across a range of array sizes, since one passing test on one small array is not much evidence. Those timings become the baseline that Task 3 measures the other two algorithms against.

This time we check three things with [the @test macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test): that the sorted result matches [Julia's `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Base.sort), that the non-mutating wrapper leaves its input untouched, and that [the mutating `bubblesort!(...)` version](src/Compute.jl) sorts in place and returns the same vector it was given.

So, did we pass the tests?

In [ ]:
let

    # initialize -
    N = 1000; # number of elements in the random vector
    arr = rand(N); # random vector length N
    original = copy(arr); # keep a copy, so we can check the wrapper left arr alone

    # check: do we get the same result as the built-in sort function?
    @test sort(arr) == L3dSorting.bubblesort(arr) # if the test fails, an error is thrown!

    # check the contracts: the wrapper must not touch its input ...
    @test arr == original

    # ... and the mutating version must sort the very vector it was handed -
    mutable_demo = rand(N);
    expected = sort(mutable_demo); # sort(...) copies, so this is safe to compute first
    @test L3dSorting.bubblesort!(mutable_demo) === mutable_demo
    @test mutable_demo == expected
end

After those checks pass, [our `bubblesort(...)` implementation](src/Compute.jl) is ready to time. Now we measure how it performs as the input grows.

The benchmark loops over the sizes in `number_of_items_per_trial`, the powers of 2 from $2^{1}$ to $2^{10}$ that the Constants subsection fixes. For each size it uses [the `@benchmarkable` macro](https://juliaci.github.io/BenchmarkTools.jl/stable/reference/#BenchmarkTools.@benchmarkable-Tuple) with a `setup=` argument that draws a fresh random `Float64` vector for every sample, so the timing excludes building the data.

We store one row for each size: the size, the mean runtime, and the standard deviation. The rows go into the `bubble_sort_data::DataFrame` variable, built with [the DataFrames.jl package](https://github.com/JuliaData/DataFrames.jl):

In [ ]:
bubble_sort_data = let
    bubble_sort_data = DataFrame();
    for i ∈ eachindex(number_of_items_per_trial)
        size_of_rand_vec_to_sort = number_of_items_per_trial[i];

        # run the test with different size vectors -
        test_run = @benchmarkable L3dSorting.bubblesort(data) setup=(data=rand($(size_of_rand_vec_to_sort)));
        results = run(test_run; samples = 20, seconds = 0.25, evals = 1)

        # store the results -
        row = (
            n = size_of_rand_vec_to_sort,
            μ = mean(results.times),
            σ = std(results.times)
        );
        push!(bubble_sort_data, row)
    end
    bubble_sort_data
end

___

## Task 3: Measure Quicksort and the built-in sort against the baseline
In this task, we measure two alternatives against the quadratic baseline Task 2 established, since a baseline is only worth having once something is compared with it. The alternatives are the recursive Quicksort from the lecture and Julia's own sorting function. We start with Quicksort, checking that our implementation matches Julia's result before timing it.

> As in Task 2, [the @test macro](https://docs.julialang.org/en/v1/stdlib/Test/#Test.@test) is the check and [Julia's `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Base.sort) supplies the reference ordering, so the two implementations are compared against the same reference. The input is a random `Float64` vector, so duplicate values are very unlikely. That means this tests [our `quicksort(...)` implementation](src/Compute.jl) away from the duplicate-heavy case the algorithm notebook discusses.

Does it work?

In [ ]:
let

    # initialize -
    N = 1000; # number of elements in the random vector
    arr = rand(N); # random vector length N

    # check: do we get the same result as the built-in sort function?
    @test sort(arr) == L3dSorting.quicksort(arr) # if the test fails, an error is thrown!
end

After that test passes, our `quicksort(...)` implementation is ready to time. We benchmark it the same way as the bubble sort, over the same sizes and drawing a fresh vector from the same distribution for every sample, and the results go into the `quick_sort_data::DataFrame` variable for the plot at the end.

In [ ]:
quick_sort_data = let

    quick_sort_data = DataFrame();
    for i ∈ eachindex(number_of_items_per_trial)
        size_of_rand_vec_to_sort = number_of_items_per_trial[i];

        # run the test with different size vectors -
        test_run = @benchmarkable L3dSorting.quicksort(data) setup=(data=rand($(size_of_rand_vec_to_sort)));
        results = run(test_run; samples = 20, seconds = 0.25, evals = 1)

        # store the results -
        row = (
            n = size_of_rand_vec_to_sort,
            μ = mean(results.times),
            σ = std(results.times)
        );
        push!(quick_sort_data, row)
    end
    quick_sort_data
end

### Julia's built-in sort
Julia provides [several sorting algorithms and picks a default suited to the data](https://docs.julialang.org/en/v1/base/sort/#Sorting-Functions). How do our two implementations compare with it?

> __Should you write your own?__ This is another __buy versus build__ question. The usual answer is to buy, meaning use the library and get the benefit of work someone else has already optimized. Let's see whether the benchmarks support that.

We benchmark Julia's sort over the same sizes and the same distribution, and the results go into the `julia_sort_data::DataFrame` variable for the plot at the end.

In [ ]:
julia_sort_data = let
    julia_sort_data = DataFrame();
    for i ∈ eachindex(number_of_items_per_trial)
        size_of_rand_vec_to_sort = number_of_items_per_trial[i];

        # run the test with different size vectors -
        test_run = @benchmarkable sort(data) setup=(data=rand($(size_of_rand_vec_to_sort)));
        results = run(test_run; samples = 20, seconds = 0.25, evals = 1)

        # store the results -
        row = (
            n = size_of_rand_vec_to_sort,
            μ = mean(results.times),
            σ = std(results.times)
        );
        push!(julia_sort_data, row)
    end
    julia_sort_data
end

___

## Visualize
Unhide the code to see how we plotted the average runtime of each sorting method as a function of the length of the vector $n$.


In [ ]:
let
    plot(quick_sort_data[:,:n], quick_sort_data[:,:μ], label="quicksort",
        yscale=:log10, xscale=:log10, lw=3, c=:gray69, minorgrid=true, legend=:topleft)
    plot!(bubble_sort_data[:,:n], bubble_sort_data[:,:μ], label="bubblesort",
        yscale=:log10, xscale=:log10, lw=3, c=:red)
    plot!(julia_sort_data[:,:n], julia_sort_data[:,:μ], label="Julia sort",
        yscale=:log10, xscale=:log10, lw=3, c=:blue)
    xlims!(1e+0, 1.5e+3) # data runs to 2^10 = 1024
    ylims!(1e+0, 1e+7)
    xlabel!("Number of elements n", guidefontsize=18)
    ylabel!("Mean Runtime (ns)", guidefontsize=18)
end

__What the curves show:__ The numbers in the plot come from one machine and yours will differ, but the shape should not. On the shortest arrays our `bubblesort(...)` runs about as fast as the built-in sort and beats our `quicksort(...)`, because a quadratic algorithm with almost no overhead is hard to beat when $n$ is small. That reverses as $n$ grows: at $n = 1024$ [Julia's built-in `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Sorting-Functions) runs more than ten times faster than our `quicksort(...)`.

The plot shows two kinds of gap. Our `bubblesort(...)` bends upward toward its $\mathcal{O}(n^{2})$ bound while our `quicksort(...)` grows the way its $\mathcal{O}(n\log{n})$ average case predicts, so the gap between them keeps widening. That is a difference in growth rate.

Now look at the middle of the plot. From roughly $n = 16$ to $n = 256$ our `quicksort(...)` and the built-in sort should run close to parallel, a fixed multiple apart. That looks like a difference in constant factor rather than in growth rate.

The library beats us on both counts, but be careful how much you read off the plot. Over sizes this small, a linear curve and an $n\log{n}$ curve are too close to tell apart by eye, so a claim about growth rate here comes from knowing the algorithm rather than from the measurement. [Julia's `sort(...)` function](https://docs.julialang.org/en/v1/base/sort/#Sorting-Functions) does not always use the same algorithm: it inspects the element type and the input and chooses among specialized methods, including [radix-style sorts](https://en.wikipedia.org/wiki/Radix_sort) that order common numeric types without comparing pairs of elements at all.


___

## Summary
Completing bubble sort by hand, then timing three sorting methods on random data of the same sizes and distribution, separates how an algorithm scales from how fast it actually runs.

> __Key Takeaways:__
>
> * __Correctness comes before speed:__ An implementation is worth benchmarking only once it reproduces a reference result, since a timing number taken from wrong code tells you nothing. Testing before timing is what makes a runtime comparison mean what it appears to mean.
> * __Growth rate and constant factor are separate claims:__ Two implementations can share a scaling exponent and still differ by a large constant multiple in runtime. Curves that look parallel on a log-log plot tell you the exponents match over the sizes you measured, not that the two are equally fast.
> * __Buying usually beats building:__ A library routine can win on engineering and on algorithm choice at once, since it may inspect what it is handed and switch to a method your implementation does not have. That is why the gap is often wider than a constant factor, and why library code is the default choice in production.

Our bubblesort keeps up with the library on the smallest inputs and then becomes much slower as the data grows. That is what an $\mathcal{O}(n^2)$ method looks like on a plot, and it is why production code should call the library even when you have written your own.
___